# PRCP-0019 – Sales Effectiveness

## Project Goals
1. Data exploration insights – Sales effectiveness.
2. Build an ML model to predict Lead Category: High Potential / Low Potential.
3. Provide suggestions to improve sales effectiveness.

**Dataset:** 7,422 leads × 9 columns  
**Domain:** Product Sales  
**Original target/source field:** `Status`

The required fields are: Created, Product_ID, Source, Mobile, EMAIL, Sales_Agent,
Location, Delivery_Mode and Status.

In [ ]:
# Cell 2 – Imports

import os
import glob
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

In [ ]:
# Cell 3 – Locate dataset

preferred = [
    "./Data/data.csv", "./Data/sales_effectiveness.csv",
    "./data/data.csv", "./data/sales_effectiveness.csv",
    "./data.csv", "./sales_effectiveness.csv"
]

files = [f for f in preferred if os.path.isfile(f)]

if not files:
    files = []
    for pattern in ["./*.csv","./Data/*.csv","./data/*.csv"]:
        files.extend(glob.glob(pattern))

print("CSV candidates:")
for f in files:
    print(f)

print("\nIf no suitable CSV is available, use the MySQL option in Cell 4.")

CSV candidates:

If no suitable CSV is available, use the MySQL option in Cell 4.


In [ ]:
import sys
!{sys.executable} -m pip install mysql-connector-python

# Cell 4 – Load from CSV or MySQL

def read_sales_csv(path):
    required = {
        "Created","Product_ID","Source","Mobile","EMAIL",
        "Sales_Agent","Location","Delivery_Mode","Status"
    }
    for sep in [",",";"]:
        for enc in ["utf-8","latin1"]:
            try:
                temp = pd.read_csv(path, sep=sep, encoding=enc)
                if required.issubset(temp.columns):
                    return temp
            except Exception:
                pass
    return None

df = None
DATA_SOURCE = None

for f in files:
    df = read_sales_csv(f)
    if df is not None:
        DATA_SOURCE = f
        break

if df is None:
    try:
        import mysql.connector
        from getpass import getpass

        print("No suitable local CSV found.")
        print("Enter the MySQL password when prompted.")
        connection = mysql.connector.connect(
            host="<your-db-host>",
            port=3306,
            database="project_sales",
            user="<your-db-user>",
            password=getpass("MySQL password: ")
        )
        df = pd.read_sql("SELECT * FROM data", connection)
        connection.close()
        DATA_SOURCE = "MySQL project_sales.data"
    except Exception as e:
        raise RuntimeError(
            "Dataset could not be loaded. Put the project CSV beside this notebook "
            "or install mysql-connector-python and use the MySQL database."
        ) from e

print("Data source:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Cell 5 – Basic checks

required = [
    "Created","Product_ID","Source","Mobile","EMAIL",
    "Sales_Agent","Location","Delivery_Mode","Status"
]

missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise KeyError("Missing columns: " + str(missing_columns))

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data_Type"))
print("\nInfo:")
df.info()

In [ ]:
# Cell 6 – Missing values and duplicates

missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": df.isna().mean() * 100
}).sort_values("Missing_Count", ascending=False)

display(missing_report)
print("Duplicate rows:", int(df.duplicated().sum()))

In [ ]:
# Cell 7 – Cardinality

cardinality = pd.DataFrame({
    "Unique_Count": df.nunique(dropna=False),
    "Missing_Count": df.isna().sum()
}).sort_values("Unique_Count", ascending=False)

display(cardinality)

In [ ]:
# Cell 8 – Created date feature engineering

created = pd.to_datetime(df["Created"], errors="coerce")

print("Invalid Created values:", int(created.isna().sum()))
print("Date range:", created.min(), "to", created.max())

df["Created_Date"] = created.dt.date
df["Created_Hour"] = created.dt.hour
df["Created_DayOfWeek"] = created.dt.day_name()
df["Created_DayOfWeek_Num"] = created.dt.dayofweek
df["Created_Month"] = created.dt.month

display(df[[
    "Created","Created_Date","Created_Hour",
    "Created_DayOfWeek","Created_Month"
]].head())

In [ ]:
# Cell 9 – Original Status analysis

status_counts = df["Status"].astype(str).str.strip().value_counts()
display(status_counts.to_frame("Count"))

plt.figure(figsize=(12,6))
sns.countplot(data=df, y="Status", order=status_counts.index)
plt.title("Original Lead Status Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 – Create High/Low Potential target

high_status = {
    "converted",
    "potential",
    "long term",
    "in progress positive"
}

low_status = {
    "junk lead",
    "not responding",
    "just enquiry",
    "in progress negative",
    "lost",
    "open"
}

status_clean = df["Status"].astype(str).str.strip().str.lower()

df["Lead_Category"] = np.nan
df.loc[status_clean.isin(high_status), "Lead_Category"] = 1
df.loc[status_clean.isin(low_status), "Lead_Category"] = 0

unmapped = sorted(
    df.loc[df["Lead_Category"].isna(), "Status"].astype(str).unique()
)

if unmapped:
    print("Unmapped statuses:", unmapped)
    raise ValueError("Update the status mapping before modeling.")

df["Lead_Category"] = df["Lead_Category"].astype(int)
df["Lead_Category_Label"] = df["Lead_Category"].map({
    0:"Low Potential",
    1:"High Potential"
})

display(pd.crosstab(df["Status"], df["Lead_Category_Label"]))
display(df["Lead_Category_Label"].value_counts().to_frame("Count"))

In [ ]:
# Cell 11 – Target visualization

plt.figure(figsize=(7,5))
sns.countplot(data=df, x="Lead_Category_Label")
plt.title("Lead Category Distribution")
plt.xlabel("Lead Category")
plt.ylabel("Number of Leads")
plt.tight_layout()
plt.show()

print("Target percentages:")
display(
    df["Lead_Category_Label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)

In [ ]:
# Cell 12 – Product ID analysis

display(
    df.groupby("Lead_Category_Label")["Product_ID"]
    .agg(["count","mean","median","std","min","max"])
)

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Lead_Category_Label", y="Product_ID")
plt.title("Product ID vs Lead Category")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 – Source analysis

source_summary = df.groupby("Source")["Lead_Category"].agg(
    Leads="count",
    High_Potential_Rate="mean"
)
source_summary["High_Potential_Rate"] *= 100
source_summary = source_summary.sort_values(
    "High_Potential_Rate", ascending=False
)

display(source_summary)

plt.figure(figsize=(12,8))
sns.barplot(
    data=source_summary.reset_index(),
    x="High_Potential_Rate", y="Source"
)
plt.title("High-Potential Rate by Source")
plt.xlabel("High-Potential Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 – Location analysis

location_summary = df.groupby("Location")["Lead_Category"].agg(
    Leads="count",
    High_Potential_Rate="mean"
)
location_summary["High_Potential_Rate"] *= 100
location_summary = location_summary.sort_values(
    "High_Potential_Rate", ascending=False
)

display(location_summary)

plt.figure(figsize=(11,7))
sns.barplot(
    data=location_summary.reset_index(),
    x="High_Potential_Rate", y="Location"
)
plt.title("High-Potential Rate by Location")
plt.xlabel("High-Potential Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 – Delivery mode analysis

delivery_summary = df.groupby("Delivery_Mode")["Lead_Category"].agg(
    Leads="count",
    High_Potential_Rate="mean"
)
delivery_summary["High_Potential_Rate"] *= 100

display(delivery_summary.sort_values(
    "High_Potential_Rate", ascending=False
))

plt.figure(figsize=(9,5))
sns.barplot(
    data=delivery_summary.reset_index(),
    x="Delivery_Mode", y="High_Potential_Rate"
)
plt.title("High-Potential Rate by Delivery Mode")
plt.ylabel("High-Potential Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 – Sales agent analysis

agent_summary = df.groupby("Sales_Agent")["Lead_Category"].agg(
    Leads="count",
    High_Potential_Rate="mean"
)
agent_summary["High_Potential_Rate"] *= 100

display(agent_summary.sort_values(
    "High_Potential_Rate", ascending=False
))

In [ ]:
# Cell 17 – Time analysis

for col in ["Created_Hour","Created_Month","Created_DayOfWeek"]:
    summary = df.groupby(col)["Lead_Category"].agg(
        Leads="count",
        High_Potential_Rate="mean"
    )
    summary["High_Potential_Rate"] *= 100

    print("\n", col)
    display(summary)

    plt.figure(figsize=(10,5))
    sns.barplot(
        data=summary.reset_index(),
        x=col, y="High_Potential_Rate"
    )
    plt.title(f"High-Potential Rate by {col}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 18 – Numerical correlation

numeric_cols = [
    "Product_ID",
    "Created_Hour",
    "Created_DayOfWeek_Num",
    "Created_Month",
    "Lead_Category"
]

corr = df[numeric_cols].corr()
display(corr)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Task 1 – Data Analysis Report

Discuss the actual outputs above:
- target distribution and class imbalance
- source-wise lead quality
- location-wise lead quality
- delivery-mode patterns
- product differences
- sales-agent distribution
- creation time/month/day patterns
- missing values and duplicates
- high-cardinality features
- PII fields

Use the notebook outputs rather than unsupported assumptions.

# Task 2 – Predictive Model

The model predicts:

`1 = High Potential`  
`0 = Low Potential`

`Status` is not used as a feature because it is used to construct the target.

`Mobile`, `EMAIL`, and `Sales_Agent` are excluded from prediction because they are identifying/
operational fields.

Source and Location use frequency encoding. Delivery_Mode uses one-hot encoding.

In [ ]:
# Cell 21 – Prepare X and y

feature_columns = [
    "Product_ID",
    "Source",
    "Location",
    "Delivery_Mode",
    "Created_Hour",
    "Created_Month"
]

X = df[feature_columns].copy()
y = df["Lead_Category"].copy()

print("Features:", feature_columns)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Cell 22 – Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# Cell 23 – Frequency encoding fitted only on training data

def frequency_encode(train_df, test_df, columns):
    train_out = train_df.copy()
    test_out = test_df.copy()
    mappings = {}

    for col in columns:
        freq = (
            train_out[col]
            .fillna("Unknown")
            .astype(str)
            .value_counts(normalize=True)
        )

        mappings[col] = freq

        train_out[col + "_Frequency"] = (
            train_out[col]
            .fillna("Unknown")
            .astype(str)
            .map(freq)
            .fillna(0)
        )

        test_out[col + "_Frequency"] = (
            test_out[col]
            .fillna("Unknown")
            .astype(str)
            .map(freq)
            .fillna(0)
        )

    train_out.drop(columns=columns, inplace=True)
    test_out.drop(columns=columns, inplace=True)

    return train_out, test_out, mappings

frequency_columns = ["Source","Location"]

X_train_fe, X_test_fe, frequency_mappings = frequency_encode(
    X_train, X_test, frequency_columns
)

categorical_features = X_train_fe.select_dtypes(
    include=["object","category"]
).columns.tolist()

numeric_features = X_train_fe.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

In [ ]:
# Cell 24 – Preprocessing

try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", encoder)
        ]),
        categorical_features
    )
])

print("Preprocessing ready.")

In [ ]:
# Cell 25 – Define models

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Support Vector Machine": SVC(
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Naive Bayes": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "K-Nearest Neighbours": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

try:
    from xgboost import XGBClassifier

    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    print("XGBoost available.")
except ImportError:
    print("XGBoost not installed; it will be skipped safely.")

print("Models:")
for name in models:
    print("-", name)

In [ ]:
# Cell 26 – Train and evaluate models

results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, estimator in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipe.fit(X_train_fe, y_train)

    pred = pipe.predict(X_test_fe)
    prob = pipe.predict_proba(X_test_fe)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1_Score": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "PR_AUC": average_precision_score(y_test, prob)
    })

    trained_models[name] = pipe
    predictions[name] = pred
    probabilities[name] = prob

results_df = pd.DataFrame(results).sort_values(
    ["F1_Score","PR_AUC","ROC_AUC"],
    ascending=False
).reset_index(drop=True)

display(results_df)

# Model Comparison Report

Compare Accuracy, Precision, Recall, F1-score, ROC-AUC and PR-AUC.

For lead-quality classification, do not rely only on accuracy because the two classes are not
perfectly balanced. F1, recall, precision and PR-AUC provide additional information.

In [ ]:
# Cell 28 – Model comparison charts

for metric in [
    "Accuracy","Precision","Recall",
    "F1_Score","ROC_AUC","PR_AUC"
]:
    plt.figure(figsize=(10,5))
    sns.barplot(data=results_df, x=metric, y="Model")
    plt.title(f"Model Comparison – {metric}")
    plt.xlim(0,1)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 29 – Confusion matrices and reports

for name in results_df["Model"]:

    pred = predictions[name]
    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Low Potential","High Potential"],
        yticklabels=["Low Potential","High Potential"]
    )
    plt.title(f"Confusion Matrix – {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    print("\n", name)
    print(classification_report(
        y_test, pred,
        target_names=["Low Potential","High Potential"],
        zero_division=0
    ))

In [ ]:
# Cell 30 – ROC curves

plt.figure(figsize=(9,7))

for name in results_df["Model"]:
    fpr, tpr, _ = roc_curve(
        y_test, probabilities[name]
    )
    auc = roc_auc_score(
        y_test, probabilities[name]
    )
    plt.plot(
        fpr, tpr,
        label=f"{name} (AUC={auc:.3f})"
    )

plt.plot([0,1],[0,1],"--",label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 31 – Precision-Recall curves

plt.figure(figsize=(9,7))

for name in results_df["Model"]:
    precision, recall, _ = precision_recall_curve(
        y_test, probabilities[name]
    )
    ap = average_precision_score(
        y_test, probabilities[name]
    )
    plt.plot(
        recall, precision,
        label=f"{name} (AP={ap:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 32 – Candidate model

best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_models[best_model_name]
best_pred = predictions[best_model_name]
best_prob = probabilities[best_model_name]
best_metrics = results_df.iloc[0]

print("Candidate model:", best_model_name)
print(best_metrics)

In [ ]:
# Cell 33 – Feature importance when available

model = best_pipeline.named_steps["model"]

if hasattr(model, "feature_importances_"):

    names = (
        best_pipeline
        .named_steps["preprocessor"]
        .get_feature_names_out()
    )

    importance = pd.DataFrame({
        "Feature": names,
        "Importance": model.feature_importances_
    }).sort_values(
        "Importance", ascending=False
    )

    display(importance.head(20))

    plt.figure(figsize=(10,7))
    sns.barplot(
        data=importance.head(20),
        x="Importance", y="Feature"
    )
    plt.title(f"Top Features – {best_model_name}")
    plt.tight_layout()
    plt.show()

else:
    print("Feature importance is not directly available for this model.")

In [ ]:
# Cell 34 – Lead scoring output

scored_leads = X_test.copy()

scored_leads["Actual_Lead_Category"] = y_test.to_numpy()
scored_leads["High_Potential_Probability"] = best_prob
scored_leads["Predicted_Lead_Category"] = np.where(
    best_pred == 1,
    "High Potential",
    "Low Potential"
)

display(
    scored_leads.sort_values(
        "High_Potential_Probability",
        ascending=False
    ).head(20)
)

# Task 3 – Suggestions to Improve Sales Effectiveness

1. Prioritize leads with higher predicted high-potential probability.
2. Compare lead quality across acquisition sources and improve low-performing channels.
3. Use location-level insights for sales-resource allocation.
4. Monitor delivery-mode patterns and investigate differences in lead quality.
5. Use creation-time patterns to plan sales-team staffing and follow-up.
6. Retrain the model as new lead outcomes become available.
7. Monitor precision, recall and actual conversion after deployment.
8. Keep PII fields out of the predictive baseline model.
9. Treat model probability as a prioritization score, not a guaranteed conversion.
10. Validate marketing changes using future campaign results.

# Report on Challenges Faced

| Challenge | Technique | Reason |
|---|---|---|
| Missing values | Imputation / explicit unknown handling | Preserve useful records |
| High-cardinality Source | Frequency encoding | Reduce dimensionality |
| High-cardinality Location | Frequency encoding | Reduce sparse features |
| Categorical Delivery_Mode | One-hot encoding | Convert categories to numeric variables |
| Date/time text | Date/time feature extraction | Make temporal information usable |
| PII fields | Excluded from modeling | Privacy and identifier concerns |
| Class imbalance | Stratified split and class weights where supported | Preserve class proportions |
| Multiple algorithms | Model comparison | Evaluate different learning approaches |
| Original Status as target source | Removed from X | Prevent target leakage |

# Limitations

- The High/Low Potential target is derived from historical Status categories.
- Historical lead behavior may change in future campaigns.
- Random train/test validation is a baseline; time-based validation can better simulate future use.
- High predicted probability does not guarantee conversion.
- Business costs and benefits should be considered when choosing the final probability threshold.

# Conclusion

**Data Loading → Data Quality → EDA → Target Engineering → Feature Engineering →
Preprocessing → Multiple ML Models → Model Comparison → Evaluation →
Lead Scoring → Sales Recommendations → Challenges → Limitations**

In [ ]:
# Cell 40 – Save outputs

os.makedirs("outputs", exist_ok=True)

results_df.to_csv(
    "outputs/PRCP-0019_model_comparison.csv",
    index=False
)

joblib.dump(
    best_pipeline,
    "outputs/PRCP-0019_sales_effectiveness_model.pkl"
)

pd.DataFrame({
    "Feature": feature_columns
}).to_csv(
    "outputs/PRCP-0019_model_features.csv",
    index=False
)

source_summary.to_csv(
    "outputs/PRCP-0019_source_analysis.csv"
)

location_summary.to_csv(
    "outputs/PRCP-0019_location_analysis.csv"
)

delivery_summary.to_csv(
    "outputs/PRCP-0019_delivery_mode_analysis.csv"
)

print("All output files saved in the outputs folder.")

In [ ]:
# Cell 41 – Final verification

print("="*70)
print("PRCP-0019 SALES EFFECTIVENESS – COMPLETED")
print("="*70)
print("Data source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
print("Candidate model:", best_model_name)
print("Accuracy :", round(float(best_metrics["Accuracy"]),4))
print("Precision:", round(float(best_metrics["Precision"]),4))
print("Recall   :", round(float(best_metrics["Recall"]),4))
print("F1-score :", round(float(best_metrics["F1_Score"]),4))
print("ROC-AUC  :", round(float(best_metrics["ROC_AUC"]),4))
print("PR-AUC   :", round(float(best_metrics["PR_AUC"]),4))
print("PII used as model features: NO")
print("Status used directly as model feature: NO")
print("="*70)